# LEGO Footprint Detector — Base Model Training (Colab)

Trains YOLOv8s on the converted **6-footprint** dataset (1x1, 1x2, 1x4, 2x2, 2x2_L, 2x3).

**Steps:** enable GPU (Runtime -> Change runtime type -> T4 GPU), then run each cell.
Upload `lego_yolo_dataset.zip` when Cell 3 asks. Training ~30-60 min on a T4.


## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable T4 in Runtime settings!')

## 2. Install Ultralytics (YOLOv8)

In [ ]:
!pip install -q ultralytics
print('ultralytics installed')

## 3. Upload the dataset zip
Upload `lego_yolo_dataset.zip` (the file exported from your Mac).

In [ ]:
from google.colab import files
import zipfile, os, glob
up = files.upload()                     # pick your dataset zip (e.g. lego_20_yolo.zip)
zname = list(up.keys())[0]
with zipfile.ZipFile(zname) as z:
    z.extractall('.')
# Auto-locate data.yaml wherever it extracted (folder name doesn't matter).
cands = glob.glob('**/data.yaml', recursive=True)
assert cands, 'No data.yaml found in the extracted zip!'
DATA = os.path.abspath(cands[0])
DATA_DIR = os.path.dirname(DATA)
print('data.yaml at:', DATA)
print(open(DATA).read())


## 4. Fix data.yaml path for Colab
(The zipped path points at your Mac; rewrite it to the Colab location.)

In [ ]:
import yaml
cfg = yaml.safe_load(open(DATA))
cfg['path'] = DATA_DIR
yaml.safe_dump(cfg, open(DATA,'w'), sort_keys=False)
print('classes:', cfg['names'])
print(open(DATA).read())


## 5. Train YOLOv8s
Rotation/flip augmentation on (bricks land at any angle top-down). ~50 epochs is a good base.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')              # small: accurate + real-time
results = model.train(
    data=DATA,
    epochs=50,
    imgsz=640,
    batch=32,
    degrees=180, flipud=0.5, fliplr=0.5,  # rotation invariance for top-down pieces
    hsv_v=0.5,                            # brightness variation (lighting robustness)
    mosaic=1.0,
    patience=15,
    project='runs', name='lego_base', exist_ok=True,
)


## 6. Validate + view metrics

In [ ]:
metrics = model.val(data=DATA)
print('mAP@0.5     :', round(float(metrics.box.map50), 4))
print('mAP@0.5:0.95:', round(float(metrics.box.map), 4))
print('precision   :', round(float(metrics.box.mp), 4))
print('recall      :', round(float(metrics.box.mr), 4))


## 7. Download the trained model
This `best.pt` is your base detector — put it at `models/lego_detector.pt` in the project.

In [ ]:
from google.colab import files
import glob, os
# Ultralytics may suffix the run dir (lego_base2/) or nest under runs/detect,
# so locate best.pt rather than hardcoding its path.
cands = sorted(glob.glob('**/best.pt', recursive=True), key=os.path.getmtime)
assert cands, 'No best.pt found — training may not have finished; check for last.pt.'
best = cands[-1]  # newest
print('downloading:', best, round(os.path.getsize(best)/1e6, 2), 'MB')
files.download(best)


## Done
- Place the downloaded `best.pt` at `models/lego_detector.pt` (overwrite the old one).
- Restart the backend so it loads the new 6-class model.
- Test in the Vision tab.
- Later: fine-tune on ~50-100 of your own webcam photos with `finetune_own.py`.
